# Spectral Transform Units & Cochlear Resonance
### Fixed Spectral Filters from Hankel Eigendecomposition

---

This notebook explores the **Spectral State Space Model (Spectral SSM)** and its optimized implementation, **Flash STU**, connecting their core mathematical objects to the biophysics of the cochlea.

### The Key Insight
While the Audio Transformer (Adieu Convolutions) *learns* its filterbank from data, the Spectral SSM uses **fixed filters** derived from the eigendecomposition of a Hankel matrix. These filters are mathematically guaranteed to capture long-range dependencies — and they bear a striking structural resemblance to the cochlea's *physically fixed* resonant modes.

### Papers & Repos
1. **Spectral State Space Models** (Agarwal et al., 2024, DeepMind) — `spectral_ssm-main/`
2. **Flash STU: Fast Spectral Transform Units** (Liu, Nguyen et al., 2024, Princeton) — `flash-stu-main/`
3. **Emerging Properties in Self-Supervised Vision Transformers** (Caron et al., 2021) — DINO
4. **Audio Transformers: Adieu Convolutions** (Verma & Berger, 2021)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

plt.style.use('default')
plt.rcParams.update({
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'grid.color': '#cccccc',
    'text.color': '#1a1a1a',
    'font.family': 'sans-serif',
    'savefig.transparent': True,
})

print("Environment ready.")

## 1. The Spectral Hankel Matrix

The foundation of the Spectral SSM is the **Hankel matrix** $Z \in \mathbb{R}^{n \times n}$:

$$Z_{i,j} = \frac{2}{(i+j)^3 - (i+j)}$$

This matrix captures the autocovariance structure of a sequence. Its eigendecomposition yields **fixed spectral filters** that form an optimal basis for representing linear dynamical systems — no learning required.

### Why This Matters for Hearing
The basilar membrane in the cochlea is a *physically fixed* resonant structure. Its frequency selectivity is determined by its mechanical properties (stiffness, mass, damping), not by neural learning. The Hankel matrix's fixed spectral filters are the mathematical analogue: they capture the natural resonant modes of a sequence, just as the cochlea captures the natural resonant modes of sound.

In [ ]:
def get_hankel_matrix(n):
    """Generate the spectral Hankel matrix (from spectral_ssm/stu_utils.py)."""
    Z = np.zeros((n, n))
    for i in range(1, n + 1):
        for j in range(1, n + 1):
            Z[i-1, j-1] = 2.0 / ((i + j)**3 - (i + j))
    return Z

# Build a 64x64 Hankel matrix
n = 64
H = get_hankel_matrix(n)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(H, cmap='RdBu_r', aspect='equal')
ax.set_title("Spectral Hankel Matrix $Z_{i,j} = 2 / [(i+j)^3 - (i+j)]$", fontsize=13)
ax.set_xlabel("Column index $j$")
ax.set_ylabel("Row index $i$")
fig.colorbar(im, ax=ax, label="$Z_{i,j}$")
plt.tight_layout()
plt.show()

print(f"Hankel matrix shape: {H.shape}")
print(f"Symmetric: {np.allclose(H, H.T)}")
print(f"Condition number: {np.linalg.cond(H):.2f}")

## 2. Eigendecomposition: Fixed Spectral Filters

The top-$k$ eigenvectors of $Z$ serve as **fixed convolutional filters**. Unlike the Audio Transformer's learned 2048-dimensional FC layer, these filters are *never trained* — they are derived purely from the mathematical structure of the Hankel matrix.

Each eigenvector $\phi_i$ is a spectral filter that captures a distinct temporal scale:
- **Top eigenvectors** → slow, smooth, long-range dependencies (like sustained vowels)
- **Lower eigenvectors** → fast, oscillatory, short-range patterns (like consonant transients)

This is directly analogous to the cochlea's tonotopic map, where:
- **Apex** → low-frequency, long-wavelength resonance
- **Base** → high-frequency, short-wavelength resonance

In [ ]:
# Eigendecomposition of the Hankel matrix
eig_vals, eig_vecs = eigh(H)

# Take top k=24 eigenvectors (sorted ascending, so take from the end)
k = 24
top_vals = eig_vals[-k:][::-1]
top_vecs = eig_vecs[:, -k:][:, ::-1]

# Plot eigenvalue spectrum
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(range(1, len(eig_vals)+1), np.abs(eig_vals[::-1]), 'o-', 
             color='#1a56db', markersize=4)
ax1.axvline(k, color='#c73030', linestyle='--', label=f'Top-{k} cutoff')
ax1.set_title("Eigenvalue Spectrum of the Hankel Matrix", fontsize=13)
ax1.set_xlabel("Eigenvalue index")
ax1.set_ylabel("|$\lambda_i$| (log scale)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot top 6 eigenvectors (spectral filters)
colors = ['#1a56db', '#0e8a7d', '#c73030', '#c78a17', '#7c3aed', '#555555']
for i in range(6):
    ax2.plot(top_vecs[:, i], color=colors[i], lw=1.5, 
             label=f"$\phi_{{{i+1}}}$ ($\lambda$={top_vals[i]:.4f})", alpha=0.8)
ax2.set_title("Top-6 Eigenvectors (Fixed Spectral Filters)", fontsize=13)
ax2.set_xlabel("Sequence position")
ax2.set_ylabel("Filter weight")
ax2.legend(fontsize=8, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Cochlear Resonance vs. Spectral Filters

Let's directly compare the Hankel eigenvectors to simulated cochlear resonance modes. 

The cochlea's basilar membrane can be modeled as a bank of damped harmonic oscillators at different characteristic frequencies. The impulse response of each oscillator is:

$$h(t) = e^{-\gamma t} \sin(2\pi f_c t)$$

We'll show that the Hankel eigenvectors naturally exhibit similar oscillatory structure with varying temporal scales — without ever being trained on audio data.

In [ ]:
# Generate simulated cochlear impulse responses
t_cochlea = np.linspace(0, 1, n)
freqs = [1.5, 3.0, 6.0, 12.0, 24.0, 48.0]  # Characteristic frequencies
gamma = 3.0  # Damping

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, (ax_top, ax_bot) in enumerate(zip(axes[0], axes[1])):
    # Top row: Hankel eigenvector
    ax_top.plot(top_vecs[:, idx], color=colors[idx], lw=2)
    ax_top.set_title(f"Hankel $\phi_{{{idx+1}}}$", fontsize=11, color=colors[idx])
    ax_top.set_ylabel("Weight")
    ax_top.grid(True, alpha=0.3)
    
    # Bottom row: Cochlear impulse response
    h_cochlea = np.exp(-gamma * t_cochlea) * np.sin(2 * np.pi * freqs[idx] * t_cochlea)
    ax_bot.plot(t_cochlea, h_cochlea, color=colors[idx], lw=2, linestyle='--')
    ax_bot.set_title(f"Cochlea $f_c$={freqs[idx]} Hz", fontsize=11, color=colors[idx])
    ax_bot.set_xlabel("Time")
    ax_bot.set_ylabel("Amplitude")
    ax_bot.grid(True, alpha=0.3)

plt.suptitle("Fixed Spectral Filters (Hankel) vs. Cochlear Impulse Responses", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Three Paradigms of Auditory Frontend

We can now place the Spectral SSM within a taxonomy of auditory frontends:

| Paradigm | Architecture | Filters | Biological Analogue |
|----------|-------------|---------|-------------------|
| **Fixed Spectral** | Spectral SSM / Flash STU | Hankel eigenvectors (never trained) | Basilar membrane (physically fixed) |
| **Learned** | Audio Transformer (Adieu Conv.) | 2048-dim FC layer (trained end-to-end) | Outer hair cell adaptation (efferent tuning) |
| **Self-Supervised** | DINO | Student-Teacher distillation | Predictive coding (illusory continuation) |

### Shape Grammar Isomorphism: *Spectral Filtering*
- **Backend (TileGym):** FFT-based convolution with fixed Hankel filters (no gradient updates).
- **Interface (Chaos Hearing):** Cochlear tonotopic map — physically fixed resonant modes.
- **Frontend (McDermott Demos):** Absolute pitch perception — the ability to identify frequencies without a reference, suggesting a fixed internal frequency template.

In [ ]:
# Simulate a simplified STU forward pass using NumPy
# This mirrors the JAX code in spectral_ssm/model.py

def stu_forward_numpy(inputs, eig_vals, eig_vecs, k=24):
    """Simplified STU forward pass.
    
    Args:
        inputs: [L, d_in] input sequence
        eig_vals: [k] top eigenvalues
        eig_vecs: [L, k] top eigenvectors
        k: number of spectral filters
    
    Returns:
        x_tilde: [L, k*d_in] spectral projection
    """
    L, d_in = inputs.shape
    
    # Step 1: Convolve input with each eigenvector (spectral projection)
    x_tilde = np.zeros((L, k, d_in))
    for ki in range(k):
        for di in range(d_in):
            conv_result = np.convolve(eig_vecs[:, ki], inputs[:, di])[:L]
            x_tilde[:, ki, di] = conv_result
    
    # Step 2: Scale by eigenvalue^0.25
    x_tilde *= np.abs(eig_vals[np.newaxis, :, np.newaxis]) ** 0.25
    
    # Step 3: Reshape to [L, k*d_in]
    return x_tilde.reshape(L, -1)

# Create a synthetic audio-like input (mix of frequencies)
L = 64
d_in = 4
t = np.linspace(0, 1, L)
inputs = np.column_stack([
    np.sin(2 * np.pi * 2 * t),   # Low freq
    np.sin(2 * np.pi * 8 * t),   # Mid freq
    np.sin(2 * np.pi * 20 * t),  # High freq
    np.random.randn(L) * 0.3     # Noise
])

# Run STU
k_use = 12
x_tilde = stu_forward_numpy(inputs, top_vals[:k_use], top_vecs[:, :k_use], k=k_use)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))

im1 = ax1.imshow(inputs.T, aspect='auto', cmap='RdBu_r', origin='lower')
ax1.set_title("Input Signal (4 channels × 64 time steps)", fontsize=13)
ax1.set_ylabel("Channel")
ax1.set_yticks(range(4))
ax1.set_yticklabels(["Low (2 Hz)", "Mid (8 Hz)", "High (20 Hz)", "Noise"])
fig.colorbar(im1, ax=ax1)

im2 = ax2.imshow(x_tilde.T, aspect='auto', cmap='RdBu_r', origin='lower')
ax2.set_title(f"Spectral Projection $\\tilde{{x}}$ ({k_use} filters × 4 channels = {k_use*4} dims)", fontsize=13)
ax2.set_xlabel("Time step")
ax2.set_ylabel("Spectral dimension")
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print(f"Input shape: {inputs.shape}")
print(f"Spectral projection shape: {x_tilde.shape}")

## 5. Conclusions

The Spectral STU reveals a **third paradigm** for auditory processing that sits between the fully learned Audio Transformer and the fully supervised traditional pipeline:

1. **Mathematical Inevitability:** The Hankel eigenvectors are determined purely by the structure of sequences themselves — they arise from the mathematics of autocovariance, not from data. Similarly, the cochlea's resonant modes arise from the physics of the basilar membrane, not from neural plasticity.

2. **Provable Robustness:** The Spectral SSM's performance is independent of the spectrum of the underlying dynamics. This mirrors the cochlea's ability to transduce *any* sound — speech, music, environmental noise — without retraining.

3. **Complementarity with Learning:** In practice, the STU combines fixed spectral filters with a small learned autoregressive component ($M_u$, $M_y$). This mirrors biology: the cochlea provides a fixed physical frontend, but the auditory cortex adds learned, adaptive processing on top.

---
### References
1. Agarwal, N., Suo, D., Chen, X., & Hazan, E. (2024). *Spectral State Space Models.* arXiv:2312.06837
2. Liu, Y.I., Nguyen, W., Devre, Y., Dogariu, E., Majumdar, A., & Hazan, E. (2024). *Flash STU: Fast Spectral Transform Units.* arXiv:2409.10489
3. Eguíluz, V.M., et al. (2000). *Essential Nonlinearities in Hearing.* Physical Review Letters.
4. Verma, P., & Berger, J. (2021). *Audio Transformers: Adieu Convolutions.* arXiv:2105.00335